# Canadian Pipeline Project — Run the Complete Workflow

This notebook acts as the project entry point. It combines the dataset-cleaning module and the interactive Mapbox mapping module to create a cleaned pipeline dataset and an HTML map.

## Workflow
1. Install required Python dependencies.
2. Set up the project folder and confirm the expected files are present.
3. Import the cleaning and mapping functions.
4. Configure source dataset paths.
5. Clean and combine the pipeline datasets.
6. Inspect the cleaned data.
7. Generate the interactive map.
8. Download the generated outputs in Google Colab.

**Important:** This notebook expects `clean_dataset.py`, `pipeline_map.py`, and the source data files to be available in the project directory. The two Python modules should contain the functions developed in the previous notebooks.


## 1. Install dependencies

Install the libraries required by the cleaning and mapping modules.

In [ ]:
%pip -q install pandas geopandas shapely requests openpyxl


## 2. Set up the project directory

In Colab, upload the project files or mount Google Drive. This example expects the following structure:

```text
project/
├── clean_dataset.py
├── pipeline_map.py
└── data/
    ├── Pipeline_Canada.csv
    ├── Pipeline_Canada2.geojson
    ├── Pipeline_Segments_BC.csv
    ├── Pipeline_AB.csv
    └── Pipeline_NB.csv
```

Set `PROJECT_DIR` to the folder containing the Python modules and the `data` folder.

In [ ]:
from pathlib import Path
import os

# In Colab, the default working directory is usually /content.
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

print(f"Project directory: {PROJECT_DIR}")
print(f"Data directory: {DATA_DIR}")


## 3. Check required project files

This cell checks that both Python modules and all five input datasets exist before starting the workflow.

In [ ]:
required_files = [
    PROJECT_DIR / "clean_dataset.py",
    PROJECT_DIR / "pipeline_map.py",
    DATA_DIR / "Pipeline_Canada.csv",
    DATA_DIR / "Pipeline_Canada2.geojson",
    DATA_DIR / "Pipeline_Segments_BC.csv",
    DATA_DIR / "Pipeline_AB.csv",
    DATA_DIR / "Pipeline_NB.csv",
]

missing_files = [path for path in required_files if not path.is_file()]

if missing_files:
    print("The following required files are missing:")
    for path in missing_files:
        print(f"  - {path}")
    raise FileNotFoundError(
        "Upload the missing project files or update PROJECT_DIR before continuing."
    )

print("All required project files were found.")


## 4. Import project functions

Import the cleaning and map-generation functions from the Python modules. If you edit either `.py` file after importing it, rerun this cell to reload the updated code.

In [ ]:
import sys
import importlib

# Ensure the project directory is available for local module imports.
project_path = str(PROJECT_DIR)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from clean_dataset import clean_pipeline_dataset
from pipeline_map import generate_pipeline_map

print("Project functions imported successfully.")


## 5. Configure source datasets

The dictionary maps descriptive dataset names to their source file paths. These names are retained in the cleaned output for provenance.

In [ ]:
file_paths = {
    "Pipeline_Canada": str(DATA_DIR / "Pipeline_Canada.csv"),
    "Pipeline_Canada2": str(DATA_DIR / "Pipeline_Canada2.geojson"),
    "Pipeline_Segments_BC": str(DATA_DIR / "Pipeline_Segments_BC.csv"),
    "Pipeline_AB": str(DATA_DIR / "Pipeline_AB.csv"),
    "Pipelines_NB": str(DATA_DIR / "Pipeline_NB.csv"),
}

for dataset_name, file_path in file_paths.items():
    print(f"{dataset_name}: {file_path}")


## 6. Clean and combine pipeline datasets

Run the cleaning function to standardize fields, normalize geometries, combine records, and export the cleaned CSV.

In [ ]:
pipelines_df = clean_pipeline_dataset(
    file_paths=file_paths,
    output_csv="Canadian_Pipelines_Clean.csv",
)

print("\nCleaned dataset shape:", pipelines_df.shape)
display(pipelines_df.head())


## 7. Inspect the cleaned dataset

Check the dataset dimensions, source distribution, and geometry availability before mapping.

In [ ]:
print("Rows:", len(pipelines_df))
print("Columns:", len(pipelines_df.columns))

print("\nRecords by source:")
display(
    pipelines_df["source_dataset"]
    .value_counts(dropna=False)
    .rename_axis("source_dataset")
    .to_frame("record_count")
)

print("\nMissing geometries:", pipelines_df["geometry"].isna().sum())


## 8. Configure Mapbox access

The map-generation module requires a Mapbox public access token. Store it as a Colab Secret named `MAPBOX_TOKEN` or as an environment variable. The `pipeline_map.py` module should read the token from the environment.

In [ ]:
# If you have not already configured the token in the runtime, set it here.
# Recommended in Colab:
# from google.colab import userdata
# os.environ["MAPBOX_TOKEN"] = userdata.get("MAPBOX_TOKEN")

if not os.environ.get("MAPBOX_TOKEN"):
    raise ValueError(
        "MAPBOX_TOKEN is not set. Add it to Colab Secrets or the environment, "
        "then rerun this cell."
    )

print("Mapbox token is configured.")


## 9. Generate the interactive pipeline map

Generate the HTML map from the cleaned DataFrame. The resulting file can be opened in a browser or displayed in the notebook.

In [ ]:
html_file = generate_pipeline_map(
    pipelines_df,
    output_html=str(PROJECT_DIR / "pipeline_map.html"),
)

print(f"Map saved to: {html_file}")


## 10. Preview the map in Colab

Display the generated HTML map in an embedded frame. If the map does not render, open or download the HTML file and check that the runtime has internet access.

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src=str(html_file), width="100%", height=700))


## 11. Download project outputs

Download the cleaned CSV and generated interactive HTML map from Colab.

In [ ]:
cleaned_csv_path = DATA_DIR / "Canadian_Pipelines_Clean.csv"
map_path = Path(html_file)

print("Cleaned CSV:", cleaned_csv_path)
print("Interactive map:", map_path)

try:
    from google.colab import files
    files.download(str(cleaned_csv_path))
    files.download(str(map_path))
except ImportError:
    print("Not running in Google Colab. The output files are saved in the project directory.")


## Troubleshooting

- **ModuleNotFoundError:** confirm that `clean_dataset.py` and `pipeline_map.py` are in `PROJECT_DIR`.
- **FileNotFoundError:** verify that the `data/` folder and all five source files are present, and update `PROJECT_DIR` if needed.
- **Mapbox token error:** configure `MAPBOX_TOKEN` in the runtime or Colab Secrets. The mapping module must read the same environment variable.
- **Map appears blank:** check internet access and verify that the Mapbox token is valid and has the required style access.
- **No pipeline lines appear:** inspect the cleaned `geometry` column and confirm that the source geometries use longitude/latitude coordinates (EPSG:4326) for display on the map.
